In [1]:
import json
import os
from pathlib import Path

import nest_asyncio
import pandas as pd
from dotenv import load_dotenv
from qdrant_client import AsyncQdrantClient

from enterprise_rag.router import route_query
from enterprise_rag.retrieval import retrieve_and_response
from enterprise_rag.generation import rag_formatted_response
from enterprise_rag.pipeline import handle_query
from enterprise_rag.evaluation import evaluate_router

load_dotenv()
nest_asyncio.apply()

qdrant_url = os.getenv("QDRANT_URL")
if qdrant_url:
    qdrant = AsyncQdrantClient(url=qdrant_url, api_key=os.getenv("QDRANT_API_KEY"))
else:
    # no QDRANT_URL configured -> run Qdrant in-process, no server or API key needed
    qdrant = AsyncQdrantClient(location=":memory:")

## Dynamic Routing

`route_query()` asks Claude to classify a user question into one of three
categories -- `ANTHROPIC_QUERY`, `10K_DOCUMENT_QUERY`, or `WEB_SEARCH` --
before any retrieval happens. `handle_query()` uses that decision to send
the query down the right path: a Qdrant vector search (with the retrieved
chunks' source and page carried through for citations) followed by a
Claude-generated, citation-backed answer.

![Dynamic routing pipeline](../assets/dynamic_routing_pipeline.svg)

| Component | Lives in |
|---|---|
| `route_query` | `enterprise_rag/router.py` |
| `retrieve_and_response` | `enterprise_rag/retrieval.py` |
| `rag_formatted_response` | `enterprise_rag/generation.py` |
| `handle_query` | `enterprise_rag/pipeline.py` |


In [2]:
user_query = "What was Uber's 2023 revenue?"
answer = await handle_query(qdrant, user_query)
answer

/Users/bahloulia/Downloads/agentic_software/Entreprise Grade RAG/.venv/lib/python3.11/site-packages/qdrant_client/async_qdrant_remote.py:231: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


Route: 10K_DOCUMENT_QUERY
Reason: User is asking about Uber's financial performance, which is covered in 10-K annual report filings.


'Found in 10-K filing.'

## Evaluating the Router

`route_query()` is an LLM classifier, so we shouldn't just trust it -- we
should measure it. Below we run it against 100 hand-labeled queries
(`notebooks/data/router_eval_queries.json`, ~34/33/33 split across the
three route categories) and score the predictions with `evaluate_router()`
(`enterprise_rag/evaluation.py`): overall accuracy, per-class
precision/recall/F1, a confusion matrix, and the individual queries it
got wrong.


In [3]:
eval_dataset = json.loads(Path("data/router_eval_queries.json").read_text())
results = evaluate_router(eval_dataset)

accuracy = results["report"]["accuracy"]
print(f"Accuracy: {accuracy:.1%} ({len(eval_dataset)} queries)")

Accuracy: 100.0% (100 queries)


In [4]:
pd.DataFrame(results["report"]).T

,precision,recall,f1-score,support
10K_DOCUMENT_QUERY,1.0,1.0,1.0,33.0
ANTHROPIC_QUERY,1.0,1.0,1.0,34.0
WEB_SEARCH,1.0,1.0,1.0,33.0
accuracy,1.0,1.0,1.0,1.0
macro avg,1.0,1.0,1.0,100.0
weighted avg,1.0,1.0,1.0,100.0


In [5]:
pd.DataFrame(
    results["confusion_matrix"],
    index=[f"true: {label}" for label in results["labels"]],
    columns=[f"pred: {label}" for label in results["labels"]],
)

,pred: 10K_DOCUMENT_QUERY,pred: ANTHROPIC_QUERY,pred: WEB_SEARCH
true: 10K_DOCUMENT_QUERY,33,0,0
true: ANTHROPIC_QUERY,0,34,0
true: WEB_SEARCH,0,0,33


In [6]:
errors_df = pd.DataFrame(
    [p for p in results["predictions"] if p["predicted"] != p["expected"]]
)
errors_df

""
